# SentinelAI — 01. Dataset Exploration & Statistical Inspection

**Unit**: Application Security and Intrusion Detection  
**Phase**: Phase 2 — Dataset Inspection & Verification  
**Target Classes**: `NORMAL`, `SQL_INJECTION`, `XSS`, `COMMAND_INJECTION`, `PATH_TRAVERSAL`  
**Objective**: Inspect actual raw HTTP application payload datasets, detect anomalies or data leakage, analyze class distributions, and establish reproducible preprocessing decisions.


In [1]:
import os
import glob
import pandas as pd
import numpy as np

# Set paths
RAW_HTTP_TRAIN = '../datasets/raw/http_params/payload_train.csv'
RAW_HTTP_TEST = '../datasets/raw/http_params/payload_test.csv'
PROCESSED_DATA = '../datasets/processed/payload_all_processed.csv'
CIC_DIR = '../datasets/raw/cic_ids2017/MachineLearningCVE/'

print('Environment initialized successfully.')


## 1. Load Raw Datasets
We load `payload_train.csv` and `payload_test.csv` from the HttpParamsDataset archive.


In [2]:
train_df = pd.read_csv(RAW_HTTP_TRAIN)
test_df = pd.read_csv(RAW_HTTP_TEST)

print(f'Raw Training Set Shape: {train_df.shape} (Rows: {train_df.shape[0]}, Columns: {train_df.shape[1]})')
print(f'Raw Testing Set Shape:  {test_df.shape} (Rows: {test_df.shape[0]}, Columns: {test_df.shape[1]})')
print('\nColumns:', list(train_df.columns))
train_df.head()


## 2. Missing Values & Duplicate Records Audit
Checking for null entries and duplicate payloads to ensure data integrity.


In [3]:
print('=== Missing Values ===')
print('Train Nulls:\n', train_df.isnull().sum())
print('\nTest Nulls:\n', test_df.isnull().sum())

print('\n=== Duplicate Payloads ===')
print('Train Duplicates:', train_df['payload'].duplicated().sum())
print('Test Duplicates: ', test_df['payload'].duplicated().sum())


## 3. Data Leakage Verification
Verify whether any payload from the test set appears in the training set.


In [4]:
train_set = set(train_df['payload'].dropna())
test_set = set(test_df['payload'].dropna())
leakage = train_set.intersection(test_set)
print(f'Intersection between Train and Test payloads: {len(leakage)} records.')
if len(leakage) == 0:
    print('VERIFIED: No data leakage detected between training and test partitions.')


## 4. Class Distribution & Imbalance Analysis
Analyzing the frequency and proportion of each attack class in both train and test splits.


In [5]:
dist_df = pd.DataFrame({
    'Train Count': train_df['attack_type'].value_counts(),
    'Train %': (train_df['attack_type'].value_counts(normalize=True) * 100).round(2),
    'Test Count': test_df['attack_type'].value_counts(),
    'Test %': (test_df['attack_type'].value_counts(normalize=True) * 100).round(2),
})
dist_df['Total Count'] = dist_df['Train Count'] + dist_df['Test Count']
dist_df['Total %'] = ((dist_df['Total Count'] / (len(train_df) + len(test_df))) * 100).round(2)
print(dist_df)


## 5. Qualitative Payload Inspection per Class
Representative examples of each attack type in the dataset.


In [6]:
for attack in train_df['attack_type'].unique():
    print(f'\n=================== Class: {attack.upper()} ===================')
    samples = train_df[train_df['attack_type'] == attack].head(3)
    for idx, r in samples.iterrows():
        print(f'[Len {r.length}] {repr(r.payload)}')


## 6. Secondary Dataset Inspection: CIC-IDS2017 (Network Flow)
> **Important Architectural Principle**:
> CIC-IDS2017 contains aggregated **network flow features** (e.g., Packet Lengths, Flow Duration, Inter-Arrival Times, Flag Counts), NOT raw application layer text payloads.
> Application-layer payload classification (HTTP attack classifier) and Network Intrusion Detection (flow telemetry) are kept strictly separate.


In [7]:
cic_files = glob.glob(CIC_DIR + '*.csv')
print(f'Total CIC-IDS2017 files present: {len(cic_files)}')
for f in cic_files:
    print(f'  - {os.path.basename(f)}')

if cic_files:
    sample_flow = pd.read_csv(cic_files[0], nrows=5)
    print(f'\nFeatures per flow record: {sample_flow.shape[1]}')
    print('Sample flow features:', list(sample_flow.columns[:8]))
    print('Label column:', sample_flow.columns[-1])


## 7. Preprocessing Decisions for Phase 3
1. **Canonical Label Mapping**:
   - `norm` -> `NORMAL`
   - `sqli` -> `SQL_INJECTION`
   - `xss` -> `XSS`
   - `path-traversal` -> `PATH_TRAVERSAL`
   - `cmdi` -> `COMMAND_INJECTION`
2. **Text Representation**:
   - Character and word n-gram TF-IDF vectorization to effectively capture syntactic injection markers (`<script>`, `' OR 1=1`, `../`, `| dir`).
3. **Class Imbalance Handling**:
   - Utilize `class_weight='balanced'` in Logistic Regression or Random Forest to prevent majority class dominance (`NORMAL` at 62%, `SQL_INJECTION` at 35%).
4. **Artifact Persistence**:
   - Serialized artifacts will be saved with `joblib` in `ai-service/app/models/attack_classifier/` (`model.joblib` and `vectorizer.joblib`).
